<a href="https://colab.research.google.com/github/krishnasai272/DeepLearning/blob/main/diseasedetect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# STEP 1: Upload ZIP Dataset
from google.colab import files
import os
import zipfile

print("📁 Please upload your dataset ZIP file:")
uploaded = files.upload()
zip_path = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("dataset")
print("✅ Dataset Extracted Successfully!")
print("📂 Inside dataset folder:", os.listdir("dataset"))
# STEP 2: Import Libraries
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
# STEP 3: Dataset Paths Configuration
train_path = "dataset/Train"

# Automatically handle directory name variations (lowercase vs capitalized)
if not os.path.exists(train_path) and os.path.exists("dataset/train"):
    train_path = "dataset/train"

# Handle optional validation folder safely
val_path = "dataset/Validation"
if not os.path.exists(val_path):
    if os.path.exists("dataset/validation"):
        val_path = "dataset/validation"
    else:
        # Fallback to Test folder if Validation folder is missing
        val_path = "dataset/Test" if os.path.exists("dataset/Test") else "dataset/test"

test_path = "dataset/Test"
if not os.path.exists(test_path) and os.path.exists("dataset/test"):
    test_path = "dataset/test"

IMG_SIZE = 128
BATCH_SIZE = 16

print(f"Using Train path: {train_path}")
print(f"Using Validation path: {val_path}")
# STEP 4: Data Preprocessing & Augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
)
val_datagen = ImageDataGenerator(rescale=1.0 / 255)
test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_data = train_datagen.flow_from_directory(
    train_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
)

val_data = val_datagen.flow_from_directory(
    val_path,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
)

num_classes = len(train_data.class_indices)
print(f"🌿 Detected {num_classes} classes: {list(train_data.class_indices.keys())}")

if os.path.exists(test_path):
    test_data = test_datagen.flow_from_directory(
        test_path,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
    )
else:
    test_data = None
# STEP 5: Build Model (Using MobileNetV2)
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet"
)

base_model.trainable = False  # Freeze pre-trained layers

x = base_model.output
x = GlobalAveragePooling2D()(x)  # Better than Flatten for MobileNet
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)

# Use softmax if multi-class classification, sigmoid if binary
output_activation = "softmax" if num_classes > 2 else "sigmoid"
loss_function = "categorical_crossentropy" if num_classes > 2 else "binary_crossentropy"

output = Dense(num_classes if num_classes > 2 else 1, activation=output_activation)(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(optimizer="adam", loss=loss_function, metrics=["accuracy"])
# STEP 6: Train Model
print("🚀 Training Started...")
epochs_num = 5  # Increase to 10+ if you want higher accuracy later

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=epochs_num,
    steps_per_epoch=max(1, train_data.samples // BATCH_SIZE),
    validation_steps=max(1, val_data.samples // BATCH_SIZE),
)

# STEP 7: Evaluate on Test Data (If available)
if test_data is not None:
    test_loss, test_acc = model.evaluate(test_data)
    print(f"📊 Test Accuracy: {test_acc * 100:.2f}%")
# STEP 8: Save Model
model.save("leaf_model.h5")
print("✅ Model Saved Successfully as 'leaf_model.h5'!")
# STEP 9: Upload Test Image & Predict
print("\n📸 Upload a Test Image to Check:")
uploaded_img = files.upload()

if len(uploaded_img) > 0:
    img_path = list(uploaded_img.keys())[0]

    # Preprocess image
    img = cv2.imread(img_path)
    if img is not None:
        img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img_normalized = img_resized / 255.0
        img_input = np.expand_dims(img_normalized, axis=0)

        # Predict
        prediction = model.predict(img_input)

        class_labels = list(train_data.class_indices.keys())

        if num_classes > 2:
            predicted_index = np.argmax(prediction[0])
            confidence = prediction[0][predicted_index] * 100
            result_label = class_labels[predicted_index]
            print(
                f"\n🌿 Result: {result_label} (Confidence: {confidence:.2f}%)"
            )
        else:
            score = prediction[0][0]
            if score > 0.5:
                print(f"\n🌿 Result: Diseased Leaf ({score * 100:.2f}%)")
            else:
                print(
                    f"\n🌿 Result: Healthy Leaf ({(1 - score) * 100:.2f}%)"
                )
    else:
        print("❌ Error reading the uploaded image file.")

📁 Please upload your dataset ZIP file:


Saving archive (6).zip to archive (6).zip
✅ Dataset Extracted Successfully!
📂 Inside dataset folder: ['Validation', 'Test', 'Train']
Using Train path: dataset/Train
Using Validation path: dataset/Validation
Found 1322 images belonging to 3 classes.
Found 60 images belonging to 3 classes.
🌿 Detected 3 classes: ['Healthy', 'Powdery', 'Rust']
Found 150 images belonging to 3 classes.
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
🚀 Training Started...
Epoch 1/5
82/82 ━━━━━━━━━━━━━━━━━━━━ 33s 327ms/step - accuracy: 0.8323 - loss: 0.4580 - val_accuracy: 0.9583 - val_loss: 0.1251
Epoch 2/5
 1/82 ━━━━━━━━━━━━━━━━━━━━ 14s 183ms/step - accuracy: 0.8125 - loss: 0.3164

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8125 - loss: 0.3164 - val_accuracy: 0.9375 - val_loss: 0.1531
Epoch 3/5
82/82 ━━━━━━━━━━━━━━━━━━━━ 25s 302ms/step - accuracy: 0.9188 - loss: 0.2215 - val_accuracy: 0.9583 - val_loss: 0.1341
Epoch 4/5
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 1.0000 - loss: 0.0981 - val_accuracy: 0.9792 - val_loss: 0.1114
Epoch 5/5
82/82 ━━━━━━━━━━━━━━━━━━━━ 24s 294ms/step - accuracy: 0.9464 - loss: 0.1661 - val_accuracy: 0.9375 - val_loss: 0.2561
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 175ms/step - accuracy: 0.9267 - loss: 0.3381


📊 Test Accuracy: 92.67%
✅ Model Saved Successfully as 'leaf_model.h5'!

📸 Upload a Test Image to Check:


Saving 11111111111.jpg to 11111111111.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

🌿 Result: Healthy (Confidence: 69.20%)
